## Setup
Imports, device detection, data loading, and model/optimizer initialization.

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

import helpers  # update when the helper is ready

def setup(batch_size, learning_rate):
    """Check for GPU and initialize device, model, loss, and optimizer."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print("Loading data...")
    train_loader, test_loader = helpers.load_data(batch_size=batch_size)

    print("Initializing model...")
    model = helpers.build_model().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    return device, model, criterion, optimizer, train_loader, test_loader


## Training Loop
Iterates over each epoch, runs forward/backward passes, and logs batch loss every 10 batches.

In [ ]:
def train(model, train_loader, criterion, optimizer, device, epochs):
    print("Starting training...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if (i + 1) % 10 == 0:
                print(f"Epoch [{epoch + 1}/{epochs}], Batch [{i + 1}/{len(train_loader)}], Loss: {running_loss / 10:.4f}")
                running_loss = 0.0


## Save & Run
Saves the trained model weights and wires everything together in `main()`.

In [ ]:
def save(model, path="object_recognition_model.pth"):
    torch.save(model.state_dict(), path)
    print(f"Training complete. Model saved as '{path}'.")


def main(epochs=10, learning_rate=0.001, batch_size=32):
    device, model, criterion, optimizer, train_loader, test_loader = setup(batch_size, learning_rate)
    train(model, train_loader, criterion, optimizer, device, epochs)
    save(model)


if __name__ == "__main__":
    main()
